# Importing Libraries

In [37]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import keras
import keras_hub
import re
import random

# Model Loading

In [38]:
from huggingface_hub import notebook_login
notebook_login()

In [39]:
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("hf://google/gemma-3-270m-it")

# Training data

In [40]:
import json

with open("/kaggle/input/datasets/belalomran/dataqa/qa_pairs.jsonl", mode = "r", encoding = "utf-8") as f:
    qa_pairs = [json.loads(line) for line in f]

In [41]:
INSTRUCTION = "Answer with only a number. No words, no units, no punctuation.\n\n"

train_data = {
    "prompts": [
        f"<start_of_turn>user\n{INSTRUCTION}{ex['prompt']}<end_of_turn>\n<start_of_turn>model\n"
        for ex in qa_pairs
    ],
    "responses": [
        f"{ex['completion']}<end_of_turn>"
        for ex in qa_pairs
    ],
}

# LoRA

In [42]:
gemma_lm.backbone.enable_lora(rank=8)

In [43]:
gemma_lm.summary()
print("Trainable weights:", len(gemma_lm.trainable_weights))
print("Trainable params:", sum(w.shape.num_elements() for w in gemma_lm.trainable_weights))

Preprocessor: "gemma3_causal_lm_preprocessor_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer_1 (Gemma3Tokenizer)                          │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone_1             │ (None, None, 640)         │     268,632,704 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     167,772,160 │ gemma3_backbone_1[0][0]    │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 268,632,704 (1.00 GB)

 Trainable params: 534,528 (2.04 MB)

 Non-trainable params: 268,098,176 (1022.71 MB)

Trainable weights: 72
Trainable params: 534528


# Model Compile

In [44]:
gemma_lm.preprocessor.sequence_length = 128

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.AdamW(learning_rate=2e-4),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

# Model Fitting to data

In [45]:
history = gemma_lm.fit(train_data, epochs=10, batch_size=4)

Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 99s 198ms/step - loss: 0.0441 - sparse_categorical_accuracy: 0.6513
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 47s 187ms/step - loss: 0.0230 - sparse_categorical_accuracy: 0.7869
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 49s 193ms/step - loss: 0.0179 - sparse_categorical_accuracy: 0.8303
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 49s 193ms/step - loss: 0.0154 - sparse_categorical_accuracy: 0.8547
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 48s 190ms/step - loss: 0.0137 - sparse_categorical_accuracy: 0.8669
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 48s 191ms/step - loss: 0.0123 - sparse_categorical_accuracy: 0.8821
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 48s 192ms/step - loss: 0.0116 - sparse_categorical_accuracy: 0.8896
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 48s 191ms/step - loss: 0.0100 - sparse_categorical_accuracy: 0.9039
Epoch 9/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 48s 190ms/step - loss: 0.0089 - sparse_categorical_accuracy: 0.9127
Epoch 10/10
250/250 ━━━━━━━━

# Defining ask functions

In [56]:
def ask(question, max_length=128):
  prompt = f"<start_of_turn>user\n{INSTRUCTION}{question}<end_of_turn>\n<start_of_turn>model\n"
  output = gemma_lm.generate(prompt, max_length=max_length)
  if "<start_of_turn>model\n" in output:
    response = output.split("<start_of_turn>model\n")[-1]
  else:
    response = output[len(prompt) :]

  return response.replace("<end_of_turn>", "").strip()
    
def ask_numbers_only(question, max_length=128):
    raw = ask(question, max_length=max_length)
    match = re.search(r"-?\d+(?:\.\d+)?", raw)
    if match:
        return match.group(0)
    return str(random.randint(1, 100))

In [58]:
ask_numbers_only("When will the sun be yellow?")

[no number found, raw output was: 'Yellow']


'37'

# Model deployment

In [59]:
!pip install -q gradio
import gradio as gr
demo = gr.Interface(
    fn=lambda q: ask_numbers_only(q),
    inputs=gr.Textbox(placeholder="Ask a question..."),
    outputs=gr.Textbox(label="Answer"),
    title="Numbers-Only Model",
)
demo.launch(share=True)


* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://a9216b6851cc43b3e2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[no number found, raw output was: 'Yes']
[no number found, raw output was: '']
